# Contents: Q-Learning

In [36]:
!pip install "gymnasium[all]"

In this notebook, you are required to implement Q-Learning Reinforcement learning algorithm for Frozen Lake Environment.

Write the code to define and train the agent.
Make sure to include a visualization of the end result in form of a video.

## Frozen Lake

Frozen lake is a toy text environment involves crossing a frozen lake from start to goal without falling into any holes by walking over the frozen lake. <br>

We can also set the lake to be slippery so that the agent does not always move in the intended direction. Here, we will only look at the non-slippery case. But you are welcome to try the slippery one.<br>

You can read more about the environment [here](https://gymnasium.farama.org/environments/toy_text/frozen_lake/).

![Frozen Lake](https://gymnasium.farama.org/_images/frozen_lake.gif)


## OpenAI Gym

[OpenAI Gym](https://www.gymlibrary.dev/) is a toolkit for developing and comparing reinforcement learning (RL) algorithms. It consists of a growing suite of environments (from simulated robots to Atari games), and a site for comparing and reproducing results. OpenAI Gym provides a diverse suite of environments that range from easy to difficult and involve many different kinds of data.

Creating and Interacting with gym environments is very simple.

```
import gym
env = gym.make("CartPole-v1")
observation, info = env.reset(seed=42)

for _ in range(1000):
    action = env.action_space.sample()
    observation, reward, done, truncated, info = env.step(action)

    if terminated or truncated:
        observation, info = env.reset()
env.close()
```

Following are the definitions of some common terminologies used.

**Reset:** Resets the environment to an initial state and returns the initial observation. <br>
**Step:** Run one timestep of the environment's dynamics.<br>
**Observation:** The observed state of the environment.<br>
**Action:** An action provided by the agent.<br>
**Reward:** The amount of reward returned as a result of taking the action.<br>
**Terminated:** Whether a terminal state (as defined under the MDP of the task) is reached.<br>
**Truncated:** Whether a truncation condition outside the scope of the MDP is satisfied. Typically a timelimit, but could also be used to indicate agent physically going out of bounds.<br>
**Info:** This contains auxiliary diagnostic information (helpful for debugging, learning, and logging).<br>
**Action Space:** This attribute gives the format of valid actions. It is of datatype Space provided by Gym. For example, if the action space is of type Discrete and gives the value Discrete(2), this means there are two valid discrete actions: 0 & 1.<br>
**Observation:** This attribute gives the format of valid observations. It is of datatype Space provided by Gym. For example, if the observation space is of type Box and the shape of the object is (4,), this denotes a valid observation will be an array of 4 numbers.<br>

Note: Previously, `terminated` and `truncated` used to be merged under one variable `done`. <br>


We will use OpenAI Gym for Frozen Lake environment.

## Creating the environment

In [37]:
import numpy as np
import gymnasium as gym
import random

In [38]:
# env = gym.make("FrozenLake-v1", is_slippery=False)
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")

### Solve here

write the code to define and train the agent:

In [39]:
#TODO What shape must the Q-table have?
print("The number of states is:", env.observation_space.n)
print("The number of actions is:", env.action_space.n)
print("The Q-table shape must be:", (env.observation_space.n, env.action_space.n))

The number of states is: 16
The number of actions is: 4
The Q-table shape must be: (np.int64(16), np.int64(4))


In [40]:
state_size = env.observation_space.n
print(state_size)

16


In [41]:
action_size = env.action_space.n
print(action_size)

4


In [42]:
# TODO: initialize the Q-table with zeros
Q_table = np.zeros((state_size, action_size))
print(Q_table)

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


In [43]:
# TODO: Identify the hyperparameters needed for Q-Learning and set reasonable values.
# epsilon = 1.0
# max_epsilon = 1.0
# min_epsilon = 0.01
# decay_rate = 0.0005
###############################################
# TODO: Identify the hyperparameters needed for Q-Learning and set reasonable values.
alpha = 0.1 # Learning rate 
gamma = 0.99 # Discount factor 
num_episodes = 1000 # Number of episodes 
max_steps_per_episode = 100 # Maximum steps per episode 

epsilon = 1.0 # Exploration rate 
max_epsilon = 1.0
min_epsilon = 0.01 # Minimum exploration rate 
decay_rate = 0.0005 # Exponential decay rate for exploration probability 

In [44]:
rewards = []
total_episodes = num_episodes ### ADDED: Assigned the value to the variable
for episode in range(total_episodes):
  state, _ = env.reset()
  step = 0
  done = False
  total_rewards = 0


  for step in range(max_steps_per_episode):
    # TODO: implement ε-greedy action selection
    # 1. Draw a random number in [0,1]
    # 2. If > epsilon: choose the greedy action (argmax from Q-table)
    # 3. Else: choose a random action from the environment (env.action_space.sample())
    ### ADDED START
    if random.uniform(0, 1) > epsilon:
        action = np.argmax(Q_table[state, :]) 
    else:
        action = env.action_space.sample()
        ### ADDED END
    new_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

    # TODO: implement the Q-learning update rule
    ### ADDED START
    # TD Target = R + gamma * max(Q(s', a'))
    td_target = reward + gamma * np.max(Q_table[new_state, :]) 
    # TD Error = TD Target - Q(s, a)
    td_error = td_target - Q_table[state, action] 
    # Q(s, a) = Q(s, a) + alpha * TD Error
    Q_table[state, action] = Q_table[state, action] + alpha * td_error 
    ### ADDED END

    total_rewards += reward 


    state = new_state

    if done:
      break
    
  epsilon = min_epsilon + (max_epsilon - min_epsilon)* np.exp(-decay_rate*episode)
  rewards.append(total_rewards)
   

print(Q_table)



[[0.936868   0.94884847 0.92148558 0.93663523]
 [0.93531238 0.         0.70458443 0.89366114]
 [0.83656229 0.27175427 0.14931526 0.30116766]
 [0.35849333 0.         0.05568314 0.09710759]
 [0.94727343 0.95873581 0.         0.93698915]
 [0.         0.         0.         0.        ]
 [0.         0.5772064  0.         0.10333132]
 [0.         0.         0.         0.        ]
 [0.95589502 0.         0.96890827 0.94536373]
 [0.9235269  0.97918418 0.96499439 0.        ]
 [0.67789337 0.98626266 0.         0.26884969]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.8958029  0.9896301  0.90681486]
 [0.91449509 0.98116758 0.99998064 0.84599657]
 [0.         0.         0.         0.        ]]


### Visualization

You are provided with some functions which will help you visualize the results as a video.
Feel free to write your own code for visualization if you prefer

In [45]:
# For visualization
# from gymnasium.wrappers.monitoring import video_recorder
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML
from IPython import display
import glob
import base64, io, os

os.environ['SDL_VIDEODRIVER']='dummy'

In [46]:
os.makedirs("video", exist_ok=True)

def show_video(env_name):
    # Function to show a video in the notebook. Do not modify.
    mp4list = glob.glob('video/*.mp4')
    if len(mp4list) > 0:
        # Find the first mp4 file in the folder if the expected file is missing
        mp4 = 'video/{}.mp4'.format(env_name)
        if not os.path.exists(mp4):
            mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display.display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

In [47]:
show_video_of_model("FrozenLake-v1")

state: 0, action: 1 4 0 False
state: 4, action: 1 8 0 False
state: 8, action: 2 9 0 False
state: 9, action: 1 13 0 False
state: 13, action: 2 14 0 False
state: 14, action: 2 15 1 True


In [48]:
show_video("FrozenLake-v1")